# 📊 Notebook 1: Data Preparation & EDA
## Consistency-Constrained Multi-Task Bengali Hate Speech Detection

**Purpose**: Download BanglaMultiHate dataset, preprocess properly, run comprehensive EDA, generate publication figures.

**Runtime**: CPU only (no GPU needed)  
**Estimated Time**: ~5 minutes

---
## 1. Environment Setup

In [1]:
# Install required packages (Kaggle has most pre-installed)
!pip install -q datasets transformers

import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, OrderedDict
import warnings
warnings.filterwarnings('ignore')

# Set high-quality figure defaults
sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# Output directories
OUTPUT_DIR = '/kaggle/working'
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
DATA_DIR = os.path.join(OUTPUT_DIR, 'data')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print('✅ Environment ready')

✅ Environment ready


---
## 2. Download Dataset from HuggingFace

**BanglaMultiHate** (aridhasan/BanglaMultiHate) — Official BLP-2025 Shared Task Dataset  
- Train: 35,522 samples  
- Dev: 5,024 samples  
- Test: 10,200 samples  
- **Total: 50,746 samples**

In [2]:
from datasets import load_dataset

print('Downloading BanglaMultiHate from HuggingFace...')
dataset = load_dataset('aridhasan/BanglaMultiHate')

print(f'\n📦 Dataset loaded successfully!')
print(f'   Splits: {list(dataset.keys())}')
for split_name, split_data in dataset.items():
    print(f'   {split_name}: {len(split_data):,} samples')
print(f'   Columns: {dataset["train"].column_names}')

README.md: 0.00B [00:00, ?B/s]

data/train.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/35522 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5024 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10200 [00:00<?, ? examples/s]


📦 Dataset loaded successfully!
   Splits: ['train', 'dev', 'test']
   train: 35,522 samples
   dev: 5,024 samples
   test: 10,200 samples
   Columns: ['id', 'comment', 'category', 'subcategory', 'type_of_hate', 'severity_of_hate', 'target_of_hate']


---
## 3. Data Preprocessing — Handling NaN/None Properly

> **CRITICAL**: In the raw dataset, non-hateful comments have `type_of_hate = "None"` and `target_of_hate = "None"` stored as the Python string `"None"`. When pandas loads this from CSV, it silently converts the string `"None"` to `NaN` (null). We must handle this conversion explicitly to avoid silent label corruption.

In [3]:
def preprocess_split(split_data, split_name):
    """
    Convert HuggingFace dataset split to cleaned pandas DataFrame.
    
    Critical: Ensures 'None' labels are stored as the STRING 'None',
    not as NaN/null, which would break label encoding.
    """
    df = split_data.to_pandas()
    
    print(f'\n--- Processing {split_name} split ({len(df):,} samples) ---')
    
    # Step 1: Check for nulls BEFORE fixing
    null_counts_before = df[['type_of_hate', 'target_of_hate', 'severity_of_hate']].isnull().sum()
    print(f'   Nulls before fix: type={null_counts_before["type_of_hate"]}, '
          f'target={null_counts_before["target_of_hate"]}, '
          f'severity={null_counts_before["severity_of_hate"]}')
    
    # Step 2: Fill NaN values with string 'None' for type and target
    df['type_of_hate'] = df['type_of_hate'].fillna('None')
    df['target_of_hate'] = df['target_of_hate'].fillna('None')
    
    # Step 3: Strip whitespace from all label columns
    for col in ['type_of_hate', 'target_of_hate', 'severity_of_hate']:
        df[col] = df[col].astype(str).str.strip()
    
    # Step 4: Verify — no nulls remaining
    null_counts_after = df[['type_of_hate', 'target_of_hate', 'severity_of_hate']].isnull().sum()
    assert null_counts_after.sum() == 0, f'Still have nulls after fix: {null_counts_after}'
    
    # Step 5: Verify label values are in expected sets
    VALID_TYPES = {'None', 'Abusive', 'Political Hate', 'Profane', 'Religious Hate', 'Sexism'}
    VALID_TARGETS = {'None', 'Individual', 'Organization', 'Community', 'Society'}
    VALID_SEVERITY = {'Little to None', 'Mild', 'Severe'}
    
    actual_types = set(df['type_of_hate'].unique())
    actual_targets = set(df['target_of_hate'].unique())
    actual_severity = set(df['severity_of_hate'].unique())
    
    assert actual_types == VALID_TYPES, f'Unexpected type labels: {actual_types - VALID_TYPES}'
    assert actual_targets == VALID_TARGETS, f'Unexpected target labels: {actual_targets - VALID_TARGETS}'
    assert actual_severity == VALID_SEVERITY, f'Unexpected severity labels: {actual_severity - VALID_SEVERITY}'
    
    # Step 6: Add text statistics
    df['char_length'] = df['comment'].str.len()
    df['word_count'] = df['comment'].str.split().str.len()
    
    # Step 7: Verify no empty comments
    empty_count = (df['comment'].isna() | (df['comment'].str.strip() == '')).sum()
    print(f'   Empty comments: {empty_count}')
    
    print(f'   ✅ {split_name} preprocessing complete')
    print(f'   Unique labels: type={len(actual_types)}, target={len(actual_targets)}, severity={len(actual_severity)}')
    
    return df

# Process all splits
df_train = preprocess_split(dataset['train'], 'train')
df_dev = preprocess_split(dataset['dev'], 'dev')
df_test = preprocess_split(dataset['test'], 'test')

print(f'\n📊 Total dataset: {len(df_train) + len(df_dev) + len(df_test):,} samples')


--- Processing train split (35,522 samples) ---
   Nulls before fix: type=0, target=0, severity=0
   Empty comments: 0
   ✅ train preprocessing complete
   Unique labels: type=6, target=5, severity=3

--- Processing dev split (5,024 samples) ---
   Nulls before fix: type=0, target=0, severity=0
   Empty comments: 0
   ✅ dev preprocessing complete
   Unique labels: type=6, target=5, severity=3

--- Processing test split (10,200 samples) ---
   Nulls before fix: type=0, target=0, severity=0
   Empty comments: 0
   ✅ test preprocessing complete
   Unique labels: type=6, target=5, severity=3

📊 Total dataset: 50,746 samples


---
## 4. Save Preprocessed Data

We save as **JSON** (not CSV) to preserve the string `"None"` labels without NaN corruption.

In [4]:
# Save as JSON to preserve 'None' string labels
for df, name in [(df_train, 'train'), (df_dev, 'dev'), (df_test, 'test')]:
    # Drop text statistics columns for clean data files
    df_clean = df.drop(columns=['char_length', 'word_count'], errors='ignore')
    
    json_path = os.path.join(DATA_DIR, f'{name}.json')
    df_clean.to_json(json_path, orient='records', force_ascii=False, indent=2)
    print(f'Saved {name}.json ({len(df):,} samples, {os.path.getsize(json_path)/1e6:.1f} MB)')

# Verify round-trip: load JSON back and check
df_verify = pd.read_json(os.path.join(DATA_DIR, 'train.json'))
assert (df_verify['type_of_hate'] == 'None').sum() == (df_train['type_of_hate'] == 'None').sum()
print(f'\n✅ JSON round-trip verified: {(df_verify["type_of_hate"] == "None").sum():,} "None" labels preserved')

Saved train.json (35,522 samples, 15.4 MB)
Saved dev.json (5,024 samples, 2.2 MB)
Saved test.json (10,200 samples, 4.4 MB)

✅ JSON round-trip verified: 19,954 "None" labels preserved


---
## 5. Canonical Label Definitions

These orderings are **fixed** and used everywhere — model, loss, dataset, evaluation.

In [5]:
# ═══════════════════════════════════════════════════════════
# CANONICAL LABEL ORDERINGS — DO NOT CHANGE
# These define the index positions used in model output heads
# ═══════════════════════════════════════════════════════════

TYPE_LABELS = ['None', 'Abusive', 'Political Hate', 'Profane', 'Religious Hate', 'Sexism']
TARGET_LABELS = ['None', 'Individual', 'Organization', 'Community', 'Society']
SEVERITY_LABELS = ['Little to None', 'Mild', 'Severe']

TYPE2IDX = {label: idx for idx, label in enumerate(TYPE_LABELS)}
TARGET2IDX = {label: idx for idx, label in enumerate(TARGET_LABELS)}
SEVERITY2IDX = {label: idx for idx, label in enumerate(SEVERITY_LABELS)}

# Key indices for consistency loss
TYPE_NONE_IDX = 0       # TYPE_LABELS[0] = 'None'
TARGET_NONE_IDX = 0     # TARGET_LABELS[0] = 'None'
SEV_LITTLE_IDX = 0      # SEVERITY_LABELS[0] = 'Little to None'
SEV_SEVERE_IDX = 2      # SEVERITY_LABELS[2] = 'Severe'

print('Label Mappings:')
print(f'  Type:     {TYPE2IDX}')
print(f'  Target:   {TARGET2IDX}')
print(f'  Severity: {SEVERITY2IDX}')
print(f'\nConsistency Loss Indices:')
print(f'  type_none_idx={TYPE_NONE_IDX}, target_none_idx={TARGET_NONE_IDX}, '
      f'sev_little_idx={SEV_LITTLE_IDX}, sev_severe_idx={SEV_SEVERE_IDX}')

Label Mappings:
  Type:     {'None': 0, 'Abusive': 1, 'Political Hate': 2, 'Profane': 3, 'Religious Hate': 4, 'Sexism': 5}
  Target:   {'None': 0, 'Individual': 1, 'Organization': 2, 'Community': 3, 'Society': 4}
  Severity: {'Little to None': 0, 'Mild': 1, 'Severe': 2}

Consistency Loss Indices:
  type_none_idx=0, target_none_idx=0, sev_little_idx=0, sev_severe_idx=2


---
## 6. Comprehensive EDA

### 6.1 Label Distribution Statistics

In [6]:
print('=' * 70)
print('LABEL DISTRIBUTION ANALYSIS')
print('=' * 70)

for col, labels, name in [('type_of_hate', TYPE_LABELS, 'Hate Type'),
                           ('target_of_hate', TARGET_LABELS, 'Target'),
                           ('severity_of_hate', SEVERITY_LABELS, 'Severity')]:
    print(f'\n--- {name} ---')
    for split_name, df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
        counts = df[col].value_counts()
        total = len(df)
        print(f'  [{split_name}]')
        for label in labels:
            c = counts.get(label, 0)
            print(f'    {label:<20s}: {c:>6d} ({c/total*100:>5.1f}%)')

LABEL DISTRIBUTION ANALYSIS

--- Hate Type ---
  [Train]
    None                :  19954 ( 56.2%)
    Abusive             :   8212 ( 23.1%)
    Political Hate      :   4227 ( 11.9%)
    Profane             :   2331 (  6.6%)
    Religious Hate      :    676 (  1.9%)
    Sexism              :    122 (  0.3%)
  [Dev]
    None                :   2898 ( 57.7%)
    Abusive             :   1113 ( 22.2%)
    Political Hate      :    574 ( 11.4%)
    Profane             :    342 (  6.8%)
    Religious Hate      :     78 (  1.6%)
    Sexism              :     19 (  0.4%)
  [Test]
    None                :   5751 ( 56.4%)
    Abusive             :   2312 ( 22.7%)
    Political Hate      :   1220 ( 12.0%)
    Profane             :    709 (  7.0%)
    Religious Hate      :    179 (  1.8%)
    Sexism              :     29 (  0.3%)

--- Target ---
  [Train]
    None                :  21190 ( 59.7%)
    Individual          :   5646 ( 15.9%)
    Organization        :   3846 ( 10.8%)
    Community     

### 6.2 Cross-Split Consistency Check

In [7]:
print('=' * 70)
print('CROSS-SPLIT LABEL DISTRIBUTION COMPARISON')
print('=' * 70)

for col, labels, name in [('type_of_hate', TYPE_LABELS, 'Hate Type'),
                           ('target_of_hate', TARGET_LABELS, 'Target'),
                           ('severity_of_hate', SEVERITY_LABELS, 'Severity')]:
    print(f'\n--- {name} (% per split) ---')
    header = f'{"Label":<20s} | {"Train":>7s} | {"Dev":>7s} | {"Test":>7s}'
    print(header)
    print('-' * len(header))
    for label in labels:
        pct_train = df_train[col].value_counts(normalize=True).get(label, 0) * 100
        pct_dev = df_dev[col].value_counts(normalize=True).get(label, 0) * 100
        pct_test = df_test[col].value_counts(normalize=True).get(label, 0) * 100
        print(f'{label:<20s} | {pct_train:>6.1f}% | {pct_dev:>6.1f}% | {pct_test:>6.1f}%')

CROSS-SPLIT LABEL DISTRIBUTION COMPARISON

--- Hate Type (% per split) ---
Label                |   Train |     Dev |    Test
--------------------------------------------------
None                 |   56.2% |   57.7% |   56.4%
Abusive              |   23.1% |   22.2% |   22.7%
Political Hate       |   11.9% |   11.4% |   12.0%
Profane              |    6.6% |    6.8% |    7.0%
Religious Hate       |    1.9% |    1.6% |    1.8%
Sexism               |    0.3% |    0.4% |    0.3%

--- Target (% per split) ---
Label                |   Train |     Dev |    Test
--------------------------------------------------
None                 |   59.7% |   61.0% |   59.7%
Individual           |   15.9% |   15.0% |   15.4%
Organization         |   10.8% |   11.6% |   11.3%
Community            |    7.4% |    6.7% |    7.4%
Society              |    6.2% |    5.6% |    6.1%

--- Severity (% per split) ---
Label                |   Train |     Dev |    Test
-----------------------------------------------

### 6.3 Ground Truth Consistency Violation Analysis

**Rule**: If `type_of_hate = "None"` then `target_of_hate` MUST be `"None"` AND `severity_of_hate` MUST be `"Little to None"`

In [8]:
print('=' * 70)
print('GROUND TRUTH CONSISTENCY VIOLATION ANALYSIS')
print('=' * 70)

for split_name, df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
    none_type = df[df['type_of_hate'] == 'None']
    
    # Violation 1: type=None but target!=None
    v1 = none_type[none_type['target_of_hate'] != 'None']
    # Violation 2: type=None but severity!=Little to None
    v2 = none_type[none_type['severity_of_hate'] != 'Little to None']
    # Reverse: target!=None but type=None
    has_target = df[df['target_of_hate'] != 'None']
    v3 = has_target[has_target['type_of_hate'] == 'None']
    
    total_violations = len(v1) + len(v2)
    
    print(f'\n[{split_name}] ({len(df):,} samples)')
    print(f'  Type=None samples: {len(none_type):,}')
    print(f'  V1 (Type=None but Target!=None): {len(v1)}')
    print(f'  V2 (Type=None but Severity!=Little): {len(v2)}')
    print(f'  V3 (Target!=None but Type=None): {len(v3)}')
    print(f'  Total violations: {total_violations} ({total_violations/len(df)*100:.2f}%)')
    print(f'  Ground truth is {"✅ CLEAN" if total_violations == 0 else "❌ HAS VIOLATIONS"}!')

GROUND TRUTH CONSISTENCY VIOLATION ANALYSIS

[Train] (35,522 samples)
  Type=None samples: 19,954
  V1 (Type=None but Target!=None): 0
  V2 (Type=None but Severity!=Little): 0
  V3 (Target!=None but Type=None): 0
  Total violations: 0 (0.00%)
  Ground truth is ✅ CLEAN!

[Dev] (5,024 samples)
  Type=None samples: 2,898
  V1 (Type=None but Target!=None): 0
  V2 (Type=None but Severity!=Little): 0
  V3 (Target!=None but Type=None): 0
  Total violations: 0 (0.00%)
  Ground truth is ✅ CLEAN!

[Test] (10,200 samples)
  Type=None samples: 5,751
  V1 (Type=None but Target!=None): 0
  V2 (Type=None but Severity!=Little): 0
  V3 (Target!=None but Type=None): 0
  Total violations: 0 (0.00%)
  Ground truth is ✅ CLEAN!


### 6.4 Text Length Statistics

In [9]:
print('=' * 70)
print('TEXT LENGTH STATISTICS')
print('=' * 70)

for split_name, df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
    print(f'\n[{split_name}]')
    print(f'  Characters: mean={df["char_length"].mean():.0f}, '
          f'median={df["char_length"].median():.0f}, '
          f'max={df["char_length"].max()}, '
          f'P95={df["char_length"].quantile(0.95):.0f}')
    print(f'  Words:      mean={df["word_count"].mean():.0f}, '
          f'median={df["word_count"].median():.0f}, '
          f'max={df["word_count"].max()}, '
          f'P95={df["word_count"].quantile(0.95):.0f}')

TEXT LENGTH STATISTICS

[Train]
  Characters: mean=78, median=51, max=3710, P95=210
  Words:      mean=14, median=9, max=629, P95=36

[Dev]
  Characters: mean=80, median=52, max=3702, P95=209
  Words:      mean=14, median=9, max=614, P95=36

[Test]
  Characters: mean=78, median=52, max=3707, P95=204
  Words:      mean=14, median=9, max=620, P95=36


---
## 7. Publication Figures (300 DPI)

### Figure 1: Multi-Task Label Distributions

In [10]:
# Color palettes
type_colors = ['#4CAF50', '#F44336', '#FF9800', '#9C27B0', '#2196F3', '#E91E63']
target_colors = ['#4CAF50', '#FF5722', '#3F51B5', '#009688', '#795548']
severity_colors = ['#4CAF50', '#FFC107', '#F44336']

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Hate Type
type_counts = df_train['type_of_hate'].value_counts().reindex(TYPE_LABELS)
axes[0].barh(TYPE_LABELS[::-1], type_counts.values[::-1], color=type_colors[::-1],
             edgecolor='white', linewidth=0.5)
axes[0].set_title('Hate Type Distribution', fontsize=15, fontweight='bold')
axes[0].set_xlabel('Count', fontsize=12)
for i, (v, label) in enumerate(zip(type_counts.values[::-1], TYPE_LABELS[::-1])):
    axes[0].text(v + 100, i, f'{v:,} ({v/len(df_train)*100:.1f}%)', va='center', fontsize=10)

# Target
target_counts = df_train['target_of_hate'].value_counts().reindex(TARGET_LABELS)
axes[1].barh(TARGET_LABELS[::-1], target_counts.values[::-1], color=target_colors[::-1],
             edgecolor='white', linewidth=0.5)
axes[1].set_title('Target Distribution', fontsize=15, fontweight='bold')
axes[1].set_xlabel('Count', fontsize=12)
for i, (v, label) in enumerate(zip(target_counts.values[::-1], TARGET_LABELS[::-1])):
    axes[1].text(v + 100, i, f'{v:,} ({v/len(df_train)*100:.1f}%)', va='center', fontsize=10)

# Severity
sev_counts = df_train['severity_of_hate'].value_counts().reindex(SEVERITY_LABELS)
axes[2].barh(SEVERITY_LABELS[::-1], sev_counts.values[::-1], color=severity_colors[::-1],
             edgecolor='white', linewidth=0.5)
axes[2].set_title('Severity Distribution', fontsize=15, fontweight='bold')
axes[2].set_xlabel('Count', fontsize=12)
for i, (v, label) in enumerate(zip(sev_counts.values[::-1], SEVERITY_LABELS[::-1])):
    axes[2].text(v + 100, i, f'{v:,} ({v/len(df_train)*100:.1f}%)', va='center', fontsize=10)

plt.suptitle('BanglaMultiHate Dataset — Multi-Task Label Distributions (N=35,522)',
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig1_class_distributions.png'))
plt.show()
print('✅ Saved fig1_class_distributions.png')

✅ Saved fig1_class_distributions.png


### Figure 2: Co-occurrence Heatmaps

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap 1: Type x Severity
ct1 = pd.crosstab(df_train['type_of_hate'], df_train['severity_of_hate'])
ct1 = ct1.reindex(index=TYPE_LABELS, columns=SEVERITY_LABELS, fill_value=0)
sns.heatmap(ct1, annot=True, fmt='d', cmap='YlOrRd', ax=axes[0],
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Count'})
axes[0].set_title('Hate Type × Severity', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Severity', fontsize=12)
axes[0].set_ylabel('Hate Type', fontsize=12)

# Heatmap 2: Type x Target
ct2 = pd.crosstab(df_train['type_of_hate'], df_train['target_of_hate'])
ct2 = ct2.reindex(index=TYPE_LABELS, columns=TARGET_LABELS, fill_value=0)
sns.heatmap(ct2, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Count'})
axes[1].set_title('Hate Type × Target', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Target', fontsize=12)
axes[1].set_ylabel('Hate Type', fontsize=12)

plt.suptitle('Multi-Task Label Co-occurrence Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig2_cooccurrence_heatmaps.png'))
plt.show()
print('✅ Saved fig2_cooccurrence_heatmaps.png')

✅ Saved fig2_cooccurrence_heatmaps.png


### Figure 3: Text Length Distribution

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Character length histogram
axes[0].hist(df_train['char_length'].clip(upper=500), bins=60, color='#2196F3',
             alpha=0.85, edgecolor='white')
axes[0].axvline(df_train['char_length'].median(), color='#F44336', linestyle='--', linewidth=2,
                label=f'Median: {df_train["char_length"].median():.0f}')
axes[0].axvline(df_train['char_length'].quantile(0.95), color='#FF9800', linestyle='--', linewidth=2,
                label=f'P95: {df_train["char_length"].quantile(0.95):.0f}')
axes[0].set_title('Character Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Length (clipped at 500)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend(fontsize=10)

# Word count histogram
axes[1].hist(df_train['word_count'].clip(upper=80), bins=50, color='#4CAF50',
             alpha=0.85, edgecolor='white')
axes[1].axvline(df_train['word_count'].median(), color='#F44336', linestyle='--', linewidth=2,
                label=f'Median: {df_train["word_count"].median():.0f}')
axes[1].axvline(df_train['word_count'].quantile(0.95), color='#FF9800', linestyle='--', linewidth=2,
                label=f'P95: {df_train["word_count"].quantile(0.95):.0f}')
axes[1].set_title('Word Count Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Word Count (clipped at 80)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend(fontsize=10)

plt.suptitle('Comment Length Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig3_text_length_distribution.png'))
plt.show()
print('✅ Saved fig3_text_length_distribution.png')

✅ Saved fig3_text_length_distribution.png


### Figure 4: Label Distribution by Content Category

In [13]:
# Top 8 categories by frequency
top_cats = df_train['category'].value_counts().head(8).index.tolist()
df_top = df_train[df_train['category'].isin(top_cats)]

fig, ax = plt.subplots(figsize=(14, 6))
ct = pd.crosstab(df_top['category'], df_top['type_of_hate'])
ct = ct.reindex(columns=TYPE_LABELS, fill_value=0)
ct = ct.loc[top_cats]  # preserve frequency order

ct.plot(kind='bar', stacked=True, ax=ax, color=type_colors, edgecolor='white', linewidth=0.5)
ax.set_title('Hate Type Distribution by Content Category (Top 8)', fontsize=14, fontweight='bold')
ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(title='Hate Type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'fig4_category_hate_type.png'))
plt.show()
print('✅ Saved fig4_category_hate_type.png')

✅ Saved fig4_category_hate_type.png


---
## 8. Compute Focal Loss α Weights

In [14]:
import torch

def compute_focal_weights(df, col, label_order):
    """Compute inverse-frequency class weights: w_c = N / (C * n_c)"""
    counts = df[col].value_counts()
    total = len(df)
    n_classes = len(label_order)
    weights = []
    for label in label_order:
        c = counts.get(label, 1)
        w = total / (n_classes * c)
        weights.append(w)
    return torch.FloatTensor(weights)

print('=' * 70)
print('FOCAL LOSS ALPHA WEIGHTS (Inverse Frequency)')
print('=' * 70)

type_weights = compute_focal_weights(df_train, 'type_of_hate', TYPE_LABELS)
target_weights = compute_focal_weights(df_train, 'target_of_hate', TARGET_LABELS)
sev_weights = compute_focal_weights(df_train, 'severity_of_hate', SEVERITY_LABELS)

for name, labels, weights in [('Hate Type', TYPE_LABELS, type_weights),
                                ('Target', TARGET_LABELS, target_weights),
                                ('Severity', SEVERITY_LABELS, sev_weights)]:
    print(f'\n--- {name} ---')
    for label, w in zip(labels, weights):
        print(f'  {label:<20s}: alpha = {w:.4f}')

# Save weights for training notebook
weights_dict = {
    'type_weights': type_weights.tolist(),
    'target_weights': target_weights.tolist(),
    'severity_weights': sev_weights.tolist(),
    'type_labels': TYPE_LABELS,
    'target_labels': TARGET_LABELS,
    'severity_labels': SEVERITY_LABELS,
}
with open(os.path.join(DATA_DIR, 'focal_weights.json'), 'w') as f:
    json.dump(weights_dict, f, indent=2)
print('\n✅ Saved focal_weights.json')

FOCAL LOSS ALPHA WEIGHTS (Inverse Frequency)

--- Hate Type ---
  None                : alpha = 0.2967
  Abusive             : alpha = 0.7209
  Political Hate      : alpha = 1.4006
  Profane             : alpha = 2.5398
  Religious Hate      : alpha = 8.7579
  Sexism              : alpha = 48.5273

--- Target ---
  None                : alpha = 0.3353
  Individual          : alpha = 1.2583
  Organization        : alpha = 1.8472
  Community           : alpha = 2.6962
  Society             : alpha = 3.2220

--- Severity ---
  Little to None      : alpha = 0.5041
  Mild                : alpha = 1.7278
  Severe              : alpha = 2.2858

✅ Saved focal_weights.json


---
## 9. Dataset Statistics Summary (for Paper Table 1)

In [15]:
print('=' * 70)
print('DATASET STATISTICS SUMMARY (for Paper Table 1)')
print('=' * 70)

print('\nTable 1: BanglaMultiHate Dataset Statistics')
print('-' * 50)
print(f'  {"Split":<10} {"Samples":>10} {"Avg Chars":>10} {"Avg Words":>10}')
print(f'  {"---":<10} {"---":>10} {"---":>10} {"---":>10}')
for name, df in [('Train', df_train), ('Dev', df_dev), ('Test', df_test)]:
    print(f'  {name:<10} {len(df):>10,} {df["char_length"].mean():>10.1f} {df["word_count"].mean():>10.1f}')
print(f'  {"Total":<10} {len(df_train)+len(df_dev)+len(df_test):>10,}')

# Class imbalance ratios
type_vc = df_train['type_of_hate'].value_counts()
target_vc = df_train['target_of_hate'].value_counts()
sev_vc = df_train['severity_of_hate'].value_counts()

print('\nClass Imbalance Ratios (Train):')
print(f'  Type:     {type_vc.max()}/{type_vc.min()} = {type_vc.max()/type_vc.min():.1f}x ({type_vc.index[0]} vs {type_vc.index[-1]})')
print(f'  Target:   {target_vc.max()}/{target_vc.min()} = {target_vc.max()/target_vc.min():.1f}x ({target_vc.index[0]} vs {target_vc.index[-1]})')
print(f'  Severity: {sev_vc.max()}/{sev_vc.min()} = {sev_vc.max()/sev_vc.min():.1f}x ({sev_vc.index[0]} vs {sev_vc.index[-1]})')

DATASET STATISTICS SUMMARY (for Paper Table 1)

Table 1: BanglaMultiHate Dataset Statistics
--------------------------------------------------
  Split         Samples  Avg Chars  Avg Words
  ---               ---        ---        ---
  Train          35,522       78.2       13.8
  Dev             5,024       79.7       14.0
  Test           10,200       78.5       13.8
  Total          50,746

Class Imbalance Ratios (Train):
  Type:     19954/122 = 163.6x (None vs Sexism)
  Target:   21190/2205 = 9.6x (None vs Society)
  Severity: 23489/5180 = 4.5x (Little to None vs Severe)


---
## 10. Save EDA Report

In [16]:
eda_report = {
    'dataset_name': 'BanglaMultiHate (aridhasan/BanglaMultiHate)',
    'splits': {
        'train': len(df_train),
        'dev': len(df_dev),
        'test': len(df_test),
        'total': len(df_train) + len(df_dev) + len(df_test)
    },
    'label_orderings': {
        'type_of_hate': TYPE_LABELS,
        'target_of_hate': TARGET_LABELS,
        'severity_of_hate': SEVERITY_LABELS
    },
    'train_distributions': {
        'type_of_hate': df_train['type_of_hate'].value_counts().to_dict(),
        'target_of_hate': df_train['target_of_hate'].value_counts().to_dict(),
        'severity_of_hate': df_train['severity_of_hate'].value_counts().to_dict(),
        'category': df_train['category'].value_counts().to_dict()
    },
    'text_stats': {
        split_name: {
            'char_mean': round(df['char_length'].mean(), 1),
            'char_median': round(float(df['char_length'].median()), 1),
            'char_max': int(df['char_length'].max()),
            'char_p95': round(float(df['char_length'].quantile(0.95)), 1),
            'word_mean': round(df['word_count'].mean(), 1),
            'word_median': round(float(df['word_count'].median()), 1),
            'word_max': int(df['word_count'].max()),
            'word_p95': round(float(df['word_count'].quantile(0.95)), 1)
        }
        for split_name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]
    },
    'consistency_violations': {'train': 0, 'dev': 0, 'test': 0},
    'class_imbalance': {
        'type_ratio': round(float(type_vc.max() / type_vc.min()), 1),
        'most_frequent_type': type_vc.index[0],
        'least_frequent_type': type_vc.index[-1]
    },
    'focal_weights': weights_dict
}

report_path = os.path.join(RESULTS_DIR, 'eda_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(eda_report, f, ensure_ascii=False, indent=2)
print(f'✅ Saved comprehensive EDA report to: {report_path}')

✅ Saved comprehensive EDA report to: /kaggle/working/results/eda_report.json


---
## 11. Sample Data Preview

In [17]:
# Show sample comments from each hate type
print('=' * 70)
print('SAMPLE COMMENTS BY HATE TYPE')
print('=' * 70)

for hate_type in TYPE_LABELS:
    subset = df_train[df_train['type_of_hate'] == hate_type]
    sample = subset.iloc[0] if len(subset) > 0 else None
    if sample is not None:
        comment_preview = sample['comment'][:120] + ('...' if len(sample['comment']) > 120 else '')
        print(f'\n[{hate_type}] (n={len(subset):,})')
        print(f'  Comment: "{comment_preview}"')
        print(f'  Target: {sample["target_of_hate"]}, Severity: {sample["severity_of_hate"]}')

SAMPLE COMMENTS BY HATE TYPE

[None] (n=19,954)
  Comment: "ধন্যবাদ বর্ডার গার্ড দেরকে এভাবে পাহারা দিতে হবে ভয় পেলে চলবে না তা না হলে আমাদের উপর হামলা করতে পারে"
  Target: None, Severity: Little to None

[Abusive] (n=8,212)
  Comment: "অতিরিক্ত এ নিজেকে বাদুর বানাইয়া ফেলছেন রে"
  Target: Individual, Severity: Little to None

[Political Hate] (n=4,227)
  Comment: "এরা জনগনকে হালের বলদ বানাচ্ছে কারন ধারের টাকা দিয়ে কখনও দেশের সংকট মোকাবেলা করা সম্ভব নয়"
  Target: Organization, Severity: Mild

[Profane] (n=2,331)
  Comment: "মুসলিম বাচ্চাগুলো বাচ্চা পেরে পেরে গোটা পৃথিবীর ধ্বংস করে দেবে শালা শুয়োরের বাচ্চা গুলো"
  Target: Community, Severity: Severe

[Religious Hate] (n=676)
  Comment: "ভারত নিয়ে এত কথা হয় ভারতে একটা মসজিদ ভাঙ্গা হয়েছে আর আফগানিস্তান পাকিস্তানে প্রতিদিন হাজার হাজার মসজিদে বিস্ফোরণ হয় ..."
  Target: Society, Severity: Mild

[Sexism] (n=122)
  Comment: "অযোগ্য মহিলাদের হাতে ক্ষমতা দিয়ে দেশকে রষাতলে ফেলার আগ্রহ দেশবাসীর নাই খোদ আমেরিকায় ও এখনও কোন নারী প্রেসিডেন্ট হয়ন

---
## 12. Summary

In [18]:
print('\n' + '=' * 70)
print('NOTEBOOK 1 COMPLETE — DATA PREPARATION & EDA')
print('=' * 70)
print(f'\nDataset: BanglaMultiHate (aridhasan/BanglaMultiHate)')
print(f'  Train: {len(df_train):,} | Dev: {len(df_dev):,} | Test: {len(df_test):,} | Total: {len(df_train)+len(df_dev)+len(df_test):,}')
print(f'  Tasks: 3 (Hate Type 6-cls, Target 5-cls, Severity 3-cls)')
print(f'  GT Consistency Violations: 0%  (all splits clean)')
print(f'  Max Imbalance: {type_vc.max()/type_vc.min():.1f}x (Sexism: {type_vc.min()} samples)')
print(f'\nFiles saved to /kaggle/working/:')
print(f'  data/train.json, dev.json, test.json')
print(f'  data/focal_weights.json')
print(f'  figures/fig1-fig4 (300 DPI)')
print(f'  results/eda_report.json')
print(f'\n=> Next: Run Notebook 03 (Training Experiments)')


NOTEBOOK 1 COMPLETE — DATA PREPARATION & EDA

Dataset: BanglaMultiHate (aridhasan/BanglaMultiHate)
  Train: 35,522 | Dev: 5,024 | Test: 10,200 | Total: 50,746
  Tasks: 3 (Hate Type 6-cls, Target 5-cls, Severity 3-cls)
  GT Consistency Violations: 0%  (all splits clean)
  Max Imbalance: 163.6x (Sexism: 122 samples)

Files saved to /kaggle/working/:
  data/train.json, dev.json, test.json
  data/focal_weights.json
  figures/fig1-fig4 (300 DPI)
  results/eda_report.json

=> Next: Run Notebook 03 (Training Experiments)
